# 01 — Exploratory Data Analysis
**Zomato Order Data** | Full interactive EDA

Sections:
1. Setup & Data Loading
2. Data Quality
3. Univariate Analysis
4. Bivariate Analysis
5. Multivariate Analysis
6. Temporal Analysis
7. Peak Hour Analysis
8. Outlier Detection
9. Discount Analysis
10. Customer Acquisition & Tenure
11. City-Level Deep Dive
12. Key Findings

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Load raw data ──────────────────────────────────────────────────────────
orders  = pd.read_csv('Zomato  Order Data.xlsx - Orders.csv')
customers = pd.read_csv('Zomato  Order Data.xlsx - Customer.csv')
restaurants = pd.read_csv('Zomato  Order Data.xlsx - Restaurants.csv')

# ── Clean column names ─────────────────────────────────────────────────────
orders.columns      = orders.columns.str.strip().str.lower()
customers.columns   = customers.columns.str.strip().str.lower()
restaurants.columns = restaurants.columns.str.strip().str.lower()

# ── Parse timestamps ───────────────────────────────────────────────────────
orders['order_timestamp'] = pd.to_datetime(orders['order_timestamp'], dayfirst=True)
customers['signup_time']  = pd.to_datetime(customers['signup_time'],  dayfirst=True)

# ── Fill missing discount ──────────────────────────────────────────────────
orders['discount_amount'] = orders['discount_amount'].fillna(0)

# ── Feature engineering ────────────────────────────────────────────────────
orders['order_date']    = orders['order_timestamp'].dt.date
orders['order_month']   = orders['order_timestamp'].dt.to_period('M').astype(str)
orders['order_quarter'] = orders['order_timestamp'].dt.to_period('Q').astype(str)
orders['order_year']    = orders['order_timestamp'].dt.year
orders['day_of_week']   = orders['order_timestamp'].dt.day_name()
orders['hour']          = orders['order_timestamp'].dt.hour
orders['net_revenue']   = orders['order_amount'] - orders['discount_amount']
orders['total_charge']  = orders['order_amount'] + orders['delivery_fee']
orders['is_discounted'] = orders['discount_amount'] > 0
orders['discount_pct']  = np.where(
    orders['order_amount'] > 0,
    (orders['discount_amount'] / orders['order_amount'] * 100).round(1),
    0
)

customers['signup_month'] = customers['signup_time'].dt.to_period('M').astype(str)
customers['signup_year']  = customers['signup_time'].dt.year

# ── Merge ──────────────────────────────────────────────────────────────────
full = (
    orders
    .merge(customers.rename(columns={'customer_id': 'customer_id'}),
           on='customer_id', how='left')
    .merge(restaurants, on='restaurant_id', how='left')
)

delivered = full[full['order_status'] == 'Delivered'].copy()

print(f'Orders: {len(orders):,} | Customers: {len(customers):,} | Restaurants: {len(restaurants):,}')
print(f'Merged shape: {full.shape}')
full.head(3)

---
## 2. Data Quality

In [ ]:
# ── Missing values summary ─────────────────────────────────────────────────
print('=== MISSING VALUES ===')
for name, df in [('Orders', orders), ('Customers', customers), ('Restaurants', restaurants)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    pct = (missing / len(df) * 100).round(2)
    if len(missing):
        print(f'\n{name}:')
        print(pd.DataFrame({'count': missing, 'pct%': pct}))
    else:
        print(f'\n{name}: No missing values ✓')

# ── Duplicate check ────────────────────────────────────────────────────────
print('\n=== DUPLICATES ===')
for name, df in [('Orders', orders), ('Customers', customers), ('Restaurants', restaurants)]:
    print(f'{name}: {df.duplicated().sum()} duplicate rows')

# ── Data types ────────────────────────────────────────────────────────────
print('\n=== ORDERS DATA TYPES ===')
print(orders.dtypes)

In [ ]:
# ── Numeric summary ────────────────────────────────────────────────────────
print('=== NUMERIC SUMMARY (Orders) ===')
display(orders[['order_amount','discount_amount','delivery_fee','net_revenue','total_charge']]
        .describe().round(2))

In [ ]:
# ── Missing values heatmap ─────────────────────────────────────────────────
missing_data = full.isnull().mean() * 100
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)

if len(missing_data) > 0:
    fig = px.bar(
        x=missing_data.index, y=missing_data.values,
        title='Missing Values (%) in Merged Dataset',
        labels={'x': 'Column', 'y': 'Missing %'},
        color=missing_data.values, color_continuous_scale='Reds'
    )
    fig.show()
else:
    print('No missing values in merged dataset after cleaning ✓')

---
## 3. Univariate Analysis

In [ ]:
# ── Numeric distributions: order_amount, delivery_fee, net_revenue, total_charge ──
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Order Amount (₹)', 'Delivery Fee (₹)', 'Net Revenue (₹)', 'Total Charge (₹)']
)
cols = ['order_amount', 'delivery_fee', 'net_revenue', 'total_charge']
positions = [(1,1), (1,2), (2,1), (2,2)]
colors = ['#FF6B35', '#004E89', '#2ECC71', '#9B59B6']

for col, (r, c), color in zip(cols, positions, colors):
    fig.add_trace(
        go.Histogram(x=full[col], nbinsx=40, name=col,
                     marker_color=color, opacity=0.8, showlegend=False),
        row=r, col=c
    )

fig.update_layout(title='Distribution of Monetary Variables', height=600)
fig.show()

In [ ]:
# ── Order Status distribution ──────────────────────────────────────────────
status_counts = full['order_status'].value_counts().reset_index()
status_counts.columns = ['order_status', 'count']
status_counts['pct'] = (status_counts['count'] / status_counts['count'].sum() * 100).round(1)

fig = px.pie(
    status_counts, names='order_status', values='count',
    title='Order Status Distribution',
    color_discrete_sequence=px.colors.qualitative.Set2,
    hole=0.4
)
fig.update_traces(textinfo='label+percent')
fig.show()

In [ ]:
# ── Payment mode distribution ──────────────────────────────────────────────
pm = full['payment_mode'].value_counts().reset_index()
pm.columns = ['payment_mode', 'count']

fig = px.bar(
    pm, x='payment_mode', y='count',
    title='Payment Mode Distribution',
    color='payment_mode', color_discrete_sequence=px.colors.qualitative.Pastel,
    text='count'
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# ── Day of week order volume ───────────────────────────────────────────────
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = full['day_of_week'].value_counts().reindex(day_order).reset_index()
dow.columns = ['day', 'count']

fig = px.bar(
    dow, x='day', y='count',
    title='Orders by Day of Week',
    color='count', color_continuous_scale='Blues',
    text='count'
)
fig.update_traces(textposition='outside')
fig.update_coloraxes(showscale=False)
fig.show()

In [ ]:
# ── City distribution ──────────────────────────────────────────────────────
city_col = 'city_x' if 'city_x' in full.columns else 'city'
city = full[city_col].value_counts().reset_index()
city.columns = ['city', 'count']

fig = px.bar(
    city.sort_values('count'), x='count', y='city',
    orientation='h',
    title='Orders by City',
    color='count', color_continuous_scale='Oranges',
    text='count'
)
fig.update_traces(textposition='outside')
fig.update_coloraxes(showscale=False)
fig.show()

In [ ]:
# ── Cuisine distribution ───────────────────────────────────────────────────
cuisine = full['cuisine'].value_counts().reset_index()
cuisine.columns = ['cuisine', 'count']

fig = px.bar(
    cuisine.sort_values('count'), x='count', y='cuisine',
    orientation='h',
    title='Orders by Cuisine Type',
    color='count', color_continuous_scale='Greens',
    text='count'
)
fig.update_traces(textposition='outside')
fig.update_coloraxes(showscale=False)
fig.show()

In [ ]:
# ── Restaurant avg_rating distribution ────────────────────────────────────
fig = px.histogram(
    restaurants, x='avg_rating', nbins=20,
    title='Restaurant Rating Distribution',
    labels={'avg_rating': 'Average Rating'},
    color_discrete_sequence=['#F39C12']
)
fig.add_vline(
    x=restaurants['avg_rating'].mean(),
    line_dash='dash', line_color='red',
    annotation_text=f"Mean: {restaurants['avg_rating'].mean():.2f}"
)
fig.show()

In [ ]:
# ── Acquisition channel ────────────────────────────────────────────────────
acq = full['acquisition_channel'].value_counts().reset_index()
acq.columns = ['channel', 'count']

fig = px.pie(
    acq, names='channel', values='count',
    title='Customer Acquisition Channel',
    color_discrete_sequence=px.colors.qualitative.Vivid,
    hole=0.35
)
fig.update_traces(textinfo='label+percent')
fig.show()

---
## 4. Bivariate Analysis

In [ ]:
# ── Order amount by order status ───────────────────────────────────────────
fig = px.box(
    full, x='order_status', y='order_amount',
    title='Order Amount by Order Status',
    color='order_status',
    color_discrete_sequence=px.colors.qualitative.Safe,
    points='outliers'
)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# ── Order amount by payment mode ───────────────────────────────────────────
fig = px.box(
    full, x='payment_mode', y='order_amount',
    title='Order Amount by Payment Mode',
    color='payment_mode',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    points='outliers'
)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# ── Revenue by city (delivered only) ──────────────────────────────────────
city_col = 'city_x' if 'city_x' in delivered.columns else 'city'
rev_city = (
    delivered.groupby(city_col)['net_revenue']
    .agg(['sum','mean','count'])
    .rename(columns={'sum':'total_revenue','mean':'avg_order_value','count':'orders'})
    .reset_index()
    .rename(columns={city_col: 'city'})
    .sort_values('total_revenue', ascending=False)
)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Total Revenue by City (₹)', 'Avg Order Value by City (₹)'])

fig.add_trace(
    go.Bar(x=rev_city['city'], y=rev_city['total_revenue'],
           marker_color='#E74C3C', name='Total Revenue',
           text=rev_city['total_revenue'].apply(lambda x: f'₹{x/1e6:.1f}M'),
           textposition='outside'),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=rev_city['city'], y=rev_city['avg_order_value'],
           marker_color='#2980B9', name='AOV',
           text=rev_city['avg_order_value'].apply(lambda x: f'₹{x:.0f}'),
           textposition='outside'),
    row=1, col=2
)
fig.update_layout(title='City Revenue Analysis (Delivered Orders)', height=500, showlegend=False)
fig.show()

In [ ]:
# ── Revenue by cuisine ─────────────────────────────────────────────────────
rev_cuisine = (
    delivered.groupby('cuisine')['net_revenue']
    .agg(['sum','mean','count'])
    .rename(columns={'sum':'total_revenue','mean':'avg_order_value','count':'orders'})
    .reset_index()
    .sort_values('total_revenue', ascending=True)
)

fig = px.bar(
    rev_cuisine, x='total_revenue', y='cuisine',
    orientation='h', title='Total Revenue by Cuisine (Delivered Orders)',
    color='total_revenue', color_continuous_scale='Teal',
    text=rev_cuisine['total_revenue'].apply(lambda x: f'₹{x/1e6:.1f}M')
)
fig.update_traces(textposition='outside')
fig.update_coloraxes(showscale=False)
fig.show()

In [ ]:
# ── Cancellation & Refund rate by city ────────────────────────────────────
city_col = 'city_x' if 'city_x' in full.columns else 'city'
city_status = full.groupby(city_col)['order_status'].value_counts(normalize=True).unstack(fill_value=0)
city_status = (city_status * 100).round(1).reset_index().rename(columns={city_col: 'city'})

fig = go.Figure()
if 'Cancelled' in city_status.columns:
    fig.add_trace(go.Bar(
        x=city_status['city'], y=city_status['Cancelled'],
        name='Cancellation Rate %', marker_color='#E74C3C'
    ))
if 'Refunded' in city_status.columns:
    fig.add_trace(go.Bar(
        x=city_status['city'], y=city_status['Refunded'],
        name='Refund Rate %', marker_color='#F39C12'
    ))

fig.update_layout(
    title='Cancellation & Refund Rates by City (%)',
    barmode='group', yaxis_title='Rate (%)', height=450
)
fig.show()

In [ ]:
# ── Restaurant rating vs revenue (bubble chart) ────────────────────────────
rest_perf = (
    delivered.groupby('restaurant_id')
    .agg(total_revenue=('net_revenue','sum'),
         order_count=('order_id','count'))
    .reset_index()
    .merge(restaurants[['restaurant_id','restaurant_name','avg_rating','cuisine']], on='restaurant_id')
)

fig = px.scatter(
    rest_perf, x='avg_rating', y='total_revenue',
    size='order_count', color='cuisine',
    hover_name='restaurant_name',
    hover_data={'avg_rating': True, 'total_revenue': ':,.0f', 'order_count': True},
    title='Restaurant Rating vs Revenue (bubble = order count)',
    labels={'avg_rating': 'Avg Rating', 'total_revenue': 'Total Revenue (₹)'},
    color_discrete_sequence=px.colors.qualitative.Vivid
)
fig.show()

---
## 5. Multivariate Analysis

In [ ]:
# ── Correlation heatmap of all numeric columns ─────────────────────────────
num_cols = ['order_amount', 'discount_amount', 'delivery_fee',
            'net_revenue', 'total_charge', 'discount_pct', 'avg_rating']
corr = full[num_cols].corr().round(2)

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=corr.columns.tolist(),
    y=corr.index.tolist(),
    text=corr.values,
    texttemplate='%{text}',
    colorscale='RdBu', zmid=0,
    colorbar_title='Correlation'
))
fig.update_layout(title='Correlation Heatmap of Numeric Variables', height=550)
fig.show()

In [ ]:
# ── Cuisine × City revenue heatmap ────────────────────────────────────────
city_col = 'city_x' if 'city_x' in delivered.columns else 'city'
pivot = (
    delivered.groupby(['cuisine', city_col])['net_revenue']
    .sum()
    .unstack(fill_value=0)
    .round(0)
)

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    text=pivot.values,
    texttemplate='₹%{text:,.0f}',
    colorscale='YlOrRd',
    colorbar_title='Revenue (₹)'
))
fig.update_layout(title='Revenue Heatmap: Cuisine × City (Delivered Orders)', height=550)
fig.show()

In [ ]:
# ── Payment mode × Order status heatmap ───────────────────────────────────
pm_status = (
    full.groupby(['payment_mode', 'order_status'])
    .size()
    .unstack(fill_value=0)
)

fig = go.Figure(go.Heatmap(
    z=pm_status.values,
    x=pm_status.columns.tolist(),
    y=pm_status.index.tolist(),
    text=pm_status.values,
    texttemplate='%{text:,}',
    colorscale='Blues',
    colorbar_title='Count'
))
fig.update_layout(title='Order Count: Payment Mode × Order Status', height=400)
fig.show()

In [ ]:
# ── Parallel coordinates: order_amount, discount_pct, delivery_fee, net_revenue ──
sample = full.sample(min(3000, len(full)), random_state=42)
status_map = {s: i for i, s in enumerate(full['order_status'].unique())}
sample['status_code'] = sample['order_status'].map(status_map)

fig = px.parallel_coordinates(
    sample,
    dimensions=['order_amount', 'discount_pct', 'delivery_fee', 'net_revenue', 'total_charge'],
    color='status_code',
    color_continuous_scale=px.colors.diverging.Tealrose,
    title='Parallel Coordinates: Monetary Variables by Order Status (sample 3k)'
)
fig.show()

---
## 6. Temporal Analysis

In [ ]:
# ── Monthly order volume & revenue ─────────────────────────────────────────
monthly = (
    full.groupby('order_month')
    .agg(order_count=('order_id','count'),
         net_revenue=('net_revenue','sum'))
    .reset_index()
    .sort_values('order_month')
)

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=['Monthly Order Volume', 'Monthly Net Revenue (₹)'],
    shared_xaxes=True
)
fig.add_trace(
    go.Scatter(x=monthly['order_month'], y=monthly['order_count'],
               mode='lines+markers', fill='tozeroy',
               line_color='#3498DB', name='Orders'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=monthly['order_month'], y=monthly['net_revenue'],
               mode='lines+markers', fill='tozeroy',
               line_color='#E74C3C', name='Revenue'),
    row=2, col=1
)
fig.update_layout(title='Monthly Trends', height=600, showlegend=False)
fig.show()

In [ ]:
# ── MoM revenue growth % ───────────────────────────────────────────────────
monthly['mom_growth'] = monthly['net_revenue'].pct_change() * 100
monthly['color'] = monthly['mom_growth'].apply(lambda x: 'green' if x >= 0 else 'red')

fig = go.Figure(go.Bar(
    x=monthly['order_month'][1:],
    y=monthly['mom_growth'][1:],
    marker_color=monthly['color'][1:],
    text=monthly['mom_growth'][1:].apply(lambda x: f'{x:+.1f}%'),
    textposition='outside'
))
fig.update_layout(
    title='Month-over-Month Revenue Growth (%)',
    yaxis_title='MoM Growth (%)', height=450
)
fig.show()

In [ ]:
# ── Quarterly revenue ──────────────────────────────────────────────────────
quarterly = (
    full.groupby('order_quarter')['net_revenue']
    .sum()
    .reset_index()
    .sort_values('order_quarter')
)

fig = px.bar(
    quarterly, x='order_quarter', y='net_revenue',
    title='Quarterly Net Revenue (₹)',
    color='net_revenue', color_continuous_scale='Sunset',
    text=quarterly['net_revenue'].apply(lambda x: f'₹{x/1e6:.2f}M')
)
fig.update_traces(textposition='outside')
fig.update_coloraxes(showscale=False)
fig.show()

In [ ]:
# ── Monthly new customer signups ───────────────────────────────────────────
signups = (
    customers.groupby('signup_month')
    .size()
    .reset_index(name='new_customers')
    .sort_values('signup_month')
)

fig = px.area(
    signups, x='signup_month', y='new_customers',
    title='Monthly New Customer Signups',
    labels={'signup_month': 'Month', 'new_customers': 'New Customers'},
    color_discrete_sequence=['#27AE60']
)
fig.show()

---
## 7. Peak Hour Analysis ⏰
> The `hour` column was engineered but never visualized. This section reveals ordering patterns throughout the day.

In [ ]:
# ── Hourly order volume ────────────────────────────────────────────────────
hourly = (
    full.groupby('hour')
    .agg(order_count=('order_id','count'),
         net_revenue=('net_revenue','sum'),
         avg_order_value=('order_amount','mean'))
    .reset_index()
)

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=['Order Volume by Hour of Day', 'Net Revenue by Hour of Day'],
    shared_xaxes=True
)
fig.add_trace(
    go.Bar(x=hourly['hour'], y=hourly['order_count'],
           marker_color='#3498DB', name='Orders',
           text=hourly['order_count'], textposition='outside'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=hourly['hour'], y=hourly['net_revenue'],
               mode='lines+markers', fill='tozeroy',
               line_color='#E74C3C', name='Revenue'),
    row=2, col=1
)
fig.update_xaxes(tickmode='linear', tick0=0, dtick=1, row=2, col=1)
fig.update_layout(title='Peak Hour Analysis', height=600, showlegend=False)
fig.show()

peak_hour = hourly.loc[hourly['order_count'].idxmax(), 'hour']
print(f'Peak ordering hour: {peak_hour}:00 — {peak_hour+1}:00')

In [ ]:
# ── Heatmap: Hour × Day of Week ────────────────────────────────────────────
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
heat_data = (
    full.groupby(['day_of_week','hour'])
    .size()
    .reset_index(name='order_count')
)
heat_pivot = heat_data.pivot(index='day_of_week', columns='hour', values='order_count').fillna(0)
heat_pivot = heat_pivot.reindex(day_order)

fig = go.Figure(go.Heatmap(
    z=heat_pivot.values,
    x=[f'{h}:00' for h in heat_pivot.columns],
    y=heat_pivot.index.tolist(),
    colorscale='YlOrRd',
    colorbar_title='Orders'
))
fig.update_layout(
    title='Order Volume: Hour of Day × Day of Week',
    xaxis_title='Hour', yaxis_title='Day of Week',
    height=450
)
fig.show()

In [ ]:
# ── Time-of-day segmentation ───────────────────────────────────────────────
def time_segment(hour):
    if 6 <= hour < 11:   return 'Morning (6–11)'
    elif 11 <= hour < 15: return 'Lunch (11–15)'
    elif 15 <= hour < 18: return 'Afternoon (15–18)'
    elif 18 <= hour < 22: return 'Dinner (18–22)'
    else:                 return 'Late Night (22–6)'

full['time_segment'] = full['hour'].apply(time_segment)
segment_order = ['Morning (6–11)', 'Lunch (11–15)', 'Afternoon (15–18)', 'Dinner (18–22)', 'Late Night (22–6)']

seg = (
    full.groupby('time_segment')
    .agg(orders=('order_id','count'), revenue=('net_revenue','sum'))
    .reindex(segment_order)
    .reset_index()
)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Orders by Time Segment', 'Revenue by Time Segment'])
fig.add_trace(
    go.Bar(x=seg['time_segment'], y=seg['orders'],
           marker_color='#3498DB', text=seg['orders'], textposition='outside'),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=seg['time_segment'], y=seg['revenue'],
           marker_color='#E74C3C',
           text=seg['revenue'].apply(lambda x: f'₹{x/1e6:.1f}M'), textposition='outside'),
    row=1, col=2
)
fig.update_layout(title='Time-of-Day Segmentation', height=450, showlegend=False)
fig.update_xaxes(tickangle=-15)
fig.show()

---
## 8. Outlier Detection 📊

In [ ]:
# ── IQR-based outlier detection on order_amount ────────────────────────────
q1 = full['order_amount'].quantile(0.25)
q3 = full['order_amount'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = full[(full['order_amount'] < lower_bound) | (full['order_amount'] > upper_bound)]
print(f'IQR Method | Q1: ₹{q1:.0f}  Q3: ₹{q3:.0f}  IQR: ₹{iqr:.0f}')
print(f'Bounds: [₹{lower_bound:.0f}, ₹{upper_bound:.0f}]')
print(f'Outliers: {len(outliers):,} ({len(outliers)/len(full)*100:.2f}% of orders)')

In [ ]:
# ── Box plots with outliers for all monetary columns ──────────────────────
fig = go.Figure()
for col, color in zip(
    ['order_amount', 'delivery_fee', 'discount_amount', 'net_revenue'],
    ['#3498DB', '#E74C3C', '#2ECC71', '#9B59B6']
):
    fig.add_trace(go.Box(
        y=full[col], name=col,
        marker_color=color, boxpoints='outliers'
    ))
fig.update_layout(
    title='Boxplots for Outlier Detection — Monetary Variables',
    yaxis_title='Amount (₹)', height=500
)
fig.show()

In [ ]:
# ── Z-score analysis on order_amount ──────────────────────────────────────
full['z_score_order'] = (full['order_amount'] - full['order_amount'].mean()) / full['order_amount'].std()
z_outliers = full[full['z_score_order'].abs() > 3]
print(f'Z-score (|z|>3) outliers: {len(z_outliers):,} ({len(z_outliers)/len(full)*100:.2f}%)')

fig = px.histogram(
    full, x='z_score_order', nbins=60,
    title='Z-Score Distribution of Order Amount',
    labels={'z_score_order': 'Z-Score'},
    color_discrete_sequence=['#3498DB']
)
fig.add_vline(x=3,  line_dash='dash', line_color='red',  annotation_text='z=+3')
fig.add_vline(x=-3, line_dash='dash', line_color='red',  annotation_text='z=-3')
fig.show()

---
## 9. Discount Analysis 🏷️

In [ ]:
# ── Discount usage rate overall ────────────────────────────────────────────
disc_rate = full['is_discounted'].mean() * 100
print(f'Discount Usage Rate: {disc_rate:.1f}% of all orders')
print(f'Avg Discount %  (discounted orders): {full[full["is_discounted"]]["discount_pct"].mean():.1f}%')
print(f'Avg Discount ₹  (discounted orders): ₹{full[full["is_discounted"]]["discount_amount"].mean():.0f}')

# Pie: discounted vs not
fig = px.pie(
    names=['Discounted', 'Full Price'],
    values=[full['is_discounted'].sum(), (~full['is_discounted']).sum()],
    title='Discounted vs Full-Price Orders',
    color_discrete_sequence=['#E74C3C', '#2ECC71'],
    hole=0.4
)
fig.update_traces(textinfo='label+percent')
fig.show()

In [ ]:
# ── Discount % distribution (among discounted orders) ─────────────────────
discounted = full[full['is_discounted']]

fig = px.histogram(
    discounted, x='discount_pct', nbins=30,
    title='Discount Depth Distribution (% off order amount)',
    labels={'discount_pct': 'Discount %'},
    color_discrete_sequence=['#E74C3C']
)
fig.add_vline(
    x=discounted['discount_pct'].mean(),
    line_dash='dash', line_color='black',
    annotation_text=f"Mean: {discounted['discount_pct'].mean():.1f}%"
)
fig.show()

In [ ]:
# ── Discount rate by city ──────────────────────────────────────────────────
city_col = 'city_x' if 'city_x' in full.columns else 'city'
disc_city = (
    full.groupby(city_col)['is_discounted']
    .mean()
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
    .reset_index()
)
disc_city.columns = ['city', 'discount_rate']

fig = px.bar(
    disc_city, x='city', y='discount_rate',
    title='Discount Usage Rate by City (%)',
    color='discount_rate', color_continuous_scale='Reds',
    text=disc_city['discount_rate'].apply(lambda x: f'{x}%')
)
fig.update_traces(textposition='outside')
fig.update_coloraxes(showscale=False)
fig.show()

In [ ]:
# ── Discounted vs non-discounted order amounts (violin) ────────────────────
full['discount_label'] = full['is_discounted'].map({True: 'Discounted', False: 'Full Price'})

fig = px.violin(
    full, x='discount_label', y='order_amount',
    color='discount_label',
    box=True, points='outliers',
    title='Order Amount: Discounted vs Full Price',
    color_discrete_map={'Discounted': '#E74C3C', 'Full Price': '#2ECC71'}
)
fig.update_layout(showlegend=False)
fig.show()

disc_mean   = full[full['is_discounted']]['order_amount'].mean()
nodisc_mean = full[~full['is_discounted']]['order_amount'].mean()
print(f'Avg order amount — Discounted: ₹{disc_mean:.0f}  |  Full Price: ₹{nodisc_mean:.0f}')

In [ ]:
# ── Discount rate by acquisition channel ──────────────────────────────────
disc_acq = (
    full.groupby('acquisition_channel')['is_discounted']
    .mean()
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
    .reset_index()
)
disc_acq.columns = ['channel', 'discount_rate']

fig = px.bar(
    disc_acq, x='channel', y='discount_rate',
    title='Discount Usage Rate by Acquisition Channel (%)',
    color='discount_rate', color_continuous_scale='Oranges',
    text=disc_acq['discount_rate'].apply(lambda x: f'{x}%')
)
fig.update_traces(textposition='outside')
fig.update_coloraxes(showscale=False)
fig.show()

---
## 10. Customer Acquisition & Tenure 👥

In [ ]:
# ── Customer tenure: days from signup to first order ──────────────────────
first_order = (
    full.groupby('customer_id')['order_timestamp']
    .min()
    .reset_index()
    .rename(columns={'order_timestamp': 'first_order_time'})
)
cust_tenure = customers.merge(
    first_order, left_on='customer_id', right_on='customer_id', how='left'
)
cust_tenure['days_to_first_order'] = (
    (cust_tenure['first_order_time'] - cust_tenure['signup_time'])
    .dt.days
)

print(cust_tenure['days_to_first_order'].describe().round(1))

fig = px.histogram(
    cust_tenure.dropna(subset=['days_to_first_order']),
    x='days_to_first_order', nbins=40,
    title='Days from Signup to First Order',
    labels={'days_to_first_order': 'Days'},
    color_discrete_sequence=['#27AE60']
)
fig.add_vline(
    x=cust_tenure['days_to_first_order'].median(),
    line_dash='dash', line_color='red',
    annotation_text=f"Median: {cust_tenure['days_to_first_order'].median():.0f} days"
)
fig.show()

In [ ]:
# ── Orders per customer distribution ──────────────────────────────────────
orders_per_cust = (
    full.groupby('customer_id')['order_id']
    .count()
    .reset_index(name='order_count')
)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Orders per Customer Distribution',
                                    'Cumulative % of Revenue by Customer (Lorenz)'])

# Histogram
fig.add_trace(
    go.Histogram(x=orders_per_cust['order_count'], nbinsx=20,
                 marker_color='#9B59B6', name='Customers'),
    row=1, col=1
)

# Lorenz-style curve: cumulative revenue
cust_rev = (
    full[full['order_status']=='Delivered']
    .groupby('customer_id')['net_revenue']
    .sum()
    .sort_values()
)
lorenz_x = np.linspace(0, 100, len(cust_rev))
lorenz_y = np.cumsum(cust_rev.values) / cust_rev.sum() * 100

fig.add_trace(
    go.Scatter(x=lorenz_x, y=lorenz_y, mode='lines',
               line_color='#E74C3C', name='Revenue Concentration'),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=[0,100], y=[0,100], mode='lines',
               line=dict(dash='dash', color='grey'), name='Perfect Equality'),
    row=1, col=2
)

fig.update_layout(title='Customer Order Frequency & Revenue Concentration', height=450)
fig.update_xaxes(title_text='Orders per Customer', row=1, col=1)
fig.update_xaxes(title_text='Cumulative % of Customers', row=1, col=2)
fig.update_yaxes(title_text='Cumulative % of Revenue', row=1, col=2)
fig.show()

In [ ]:
# ── Acquisition channel — order volume & AOV ──────────────────────────────
acq_perf = (
    full[full['order_status']=='Delivered']
    .groupby('acquisition_channel')
    .agg(orders=('order_id','count'), revenue=('net_revenue','sum'))
    .assign(aov=lambda d: d['revenue'] / d['orders'])
    .reset_index()
    .sort_values('revenue', ascending=False)
)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Revenue by Acquisition Channel', 'AOV by Acquisition Channel'])
fig.add_trace(
    go.Bar(x=acq_perf['acquisition_channel'], y=acq_perf['revenue'],
           marker_color='#3498DB',
           text=acq_perf['revenue'].apply(lambda x: f'₹{x/1e6:.1f}M'),
           textposition='outside'),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=acq_perf['acquisition_channel'], y=acq_perf['aov'],
           marker_color='#E67E22',
           text=acq_perf['aov'].apply(lambda x: f'₹{x:.0f}'),
           textposition='outside'),
    row=1, col=2
)
fig.update_layout(title='Performance by Acquisition Channel', height=450, showlegend=False)
fig.show()

---
## 11. City-Level Deep Dive 🌆

In [ ]:
# ── City KPI summary table ─────────────────────────────────────────────────
city_col = 'city_x' if 'city_x' in full.columns else 'city'

city_summary = full.groupby(city_col).agg(
    total_orders        = ('order_id',        'count'),
    total_revenue       = ('net_revenue',      'sum'),
    avg_order_value     = ('order_amount',     'mean'),
    avg_delivery_fee    = ('delivery_fee',     'mean'),
    avg_discount_pct    = ('discount_pct',     'mean'),
    delivery_rate       = ('order_status',     lambda x: (x == 'Delivered').mean() * 100),
    cancel_rate         = ('order_status',     lambda x: (x == 'Cancelled').mean() * 100),
    refund_rate         = ('order_status',     lambda x: (x == 'Refunded').mean() * 100),
    avg_rating          = ('avg_rating',       'mean'),
).round(2).reset_index().rename(columns={city_col: 'city'})

city_summary = city_summary.sort_values('total_revenue', ascending=False)
display(city_summary)

In [ ]:
# ── City KPI panel ─────────────────────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        'Total Orders', 'Total Revenue (₹M)',
        'Avg Order Value (₹)', 'Delivery Rate (%)',
        'Cancellation Rate (%)', 'Avg Rating'
    ]
)

metrics = [
    ('total_orders',    (1,1), '#3498DB'),
    ('total_revenue',   (1,2), '#E74C3C'),
    ('avg_order_value', (1,3), '#2ECC71'),
    ('delivery_rate',   (2,1), '#27AE60'),
    ('cancel_rate',     (2,2), '#E67E22'),
    ('avg_rating',      (2,3), '#9B59B6'),
]

for metric, (r, c), color in metrics:
    y_vals = city_summary[metric]
    if metric == 'total_revenue':
        y_vals = y_vals / 1e6
    fig.add_trace(
        go.Bar(x=city_summary['city'], y=y_vals,
               marker_color=color, showlegend=False),
        row=r, col=c
    )

fig.update_layout(title='City-Level KPI Panel', height=700)
fig.show()

---
## 12. Key Findings 🔑

In [ ]:
# ── Auto-generate key findings ─────────────────────────────────────────────
top_city    = city_summary.iloc[0]['city']
top_cuisine = rev_cuisine.sort_values('total_revenue', ascending=False).iloc[0]['cuisine']
del_rate    = (full['order_status']=='Delivered').mean()*100
can_rate    = (full['order_status']=='Cancelled').mean()*100 if 'Cancelled' in full['order_status'].values else 0
ref_rate    = (full['order_status']=='Refunded').mean()*100  if 'Refunded'  in full['order_status'].values else 0
disc_usage  = full['is_discounted'].mean()*100
top_channel = acq_perf.sort_values('revenue', ascending=False).iloc[0]['acquisition_channel']
peak_hr     = hourly.loc[hourly['order_count'].idxmax(), 'hour']
top_segment = seg.loc[seg['orders'].idxmax(), 'time_segment']

findings = f"""
╔══════════════════════════════════════════════════════════╗
║              KEY EDA FINDINGS — ZOMATO DATA              ║
╠══════════════════════════════════════════════════════════╣
║  Dataset     : {len(orders):,} orders | {len(customers):,} customers | {len(restaurants)} restaurants
║  Time Span   : {orders['order_timestamp'].min().date()} → {orders['order_timestamp'].max().date()}
╠══ Order Outcomes ════════════════════════════════════════╣
║  Delivery Rate   : {del_rate:.1f}%
║  Cancellation    : {can_rate:.1f}%
║  Refund Rate     : {ref_rate:.1f}%
╠══ Revenue Concentration ═════════════════════════════════╣
║  Top City        : {top_city}
║  Top Cuisine     : {top_cuisine}
║  Top Channel     : {top_channel}
╠══ Ordering Patterns ═════════════════════════════════════╣
║  Peak Hour       : {peak_hr}:00–{peak_hr+1}:00
║  Busiest Segment : {top_segment}
╠══ Discounting ═══════════════════════════════════════════╣
║  Discount Usage  : {disc_usage:.1f}% of orders have a discount
║  Avg Discount %  : {full[full['is_discounted']]['discount_pct'].mean():.1f}% off (among discounted)
╠══ Outliers ══════════════════════════════════════════════╣
║  IQR Outliers    : {len(outliers):,} ({len(outliers)/len(full)*100:.2f}% of orders)
╚══════════════════════════════════════════════════════════╝
"""
print(findings)